# **PREPROCESSING**

In [ ]:
import pandas as pd
import re

# 1. Memuat Dataset
file_path = 'DATA_SELFLABEL.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di jalur: {file_path}")
    exit()

# Menentukan nama kolom yang akan diproses
kolom_data = 'DATA'

# Memastikan kolom 'DATA' ada dalam DataFrame
if kolom_data not in df.columns:
    print(f"Error: Kolom '{kolom_data}' tidak ditemukan dalam file CSV.")
    exit()

## 2. Fungsi Preprocessing: Menghilangkan Karakter Khusus

def clean_special_chars(text):
    """
    Menghilangkan semua karakter selain huruf (a-z, A-Z) dan spasi.
    """
    # Mengubah teks menjadi huruf kecil (Case Folding)
    text = str(text).lower()

    # Regex: [^a-z\s] berarti 'match anything that is NOT a lowercase letter (a-z) OR a whitespace character (\s)'
    # Karakter yang cocok akan diganti dengan spasi tunggal.
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Menghilangkan spasi berlebihan yang mungkin muncul setelah penghapusan karakter
    text = re.sub(r'\s+', ' ', text).strip()

    return text

## 3. Implementasi Preprocessing pada DataFrame

# Menerapkan fungsi ke kolom 'DATA'
df['DATA_cleaned'] = df[kolom_data].apply(clean_special_chars)

## 4. Menampilkan Hasil

print("✅ Preprocessing Selesai.\n")
print("--- Data Asli (Sebelum Dibersihkan) ---")
print(df[[kolom_data]].head())
print("\n--- Data Setelah Dibersihkan ---")
print(df[['DATA_cleaned']].head())
print("-" * 40)
print(f"Jumlah baris yang diproses: {len(df)}")

✅ Preprocessing Selesai.

--- Data Asli (Sebelum Dibersihkan) ---
                                                DATA
0                                           fan dedy
1                 semoga jirayut halda jodoh sakinah
2                                           lucu tau
3              disaat halda sudah sadam masih disini
4  cewek berisik banget sok asik garing jujurly skip

--- Data Setelah Dibersihkan ---
                                        DATA_cleaned
0                                           fan dedy
1                 semoga jirayut halda jodoh sakinah
2                                           lucu tau
3              disaat halda sudah sadam masih disini
4  cewek berisik banget sok asik garing jujurly skip
----------------------------------------
Jumlah baris yang diproses: 250


## **NORMALISASI**

In [ ]:
# Lanjutan dari kode sebelumnya (dengan asumsi df sudah dimuat dan DATA_cleaned sudah ada)

# --- Kamus Normalisasi Bahasa Gaul ---
normalization_dict = {
    # I. Clean-up Awal
    'jga': 'juga', 'slu': 'selalu', 'krna': 'karena', 'keknya': 'kayaknya',
    'ntar': 'nanti', 'jgk': 'juga', 'bnr': 'benar', 'klau': 'kalau',
    'kngen': 'kangen', 'dnk': 'dong', 'enga': 'tidak', 'tp': 'tapi',
    'dr': 'dari', 'sma': 'sama', 'smpe': 'sampai', 'bkin': 'bikin',
    'brp': 'berapa', 'skrng': 'sekarang', 'yutub': 'youtube', 'pnasaran': 'penasaran',
    'ahire': 'akhirnya', 'lgi': 'lagi', 'mreka': 'mereka', 'tgl': 'tanggal',
    'iyaa': 'iya', 'ntn': 'nonton', 'nthn': 'nonton', 'klo': 'kalau',
    'cmn': 'cuma', 'dlu': 'dulu', 'brti': 'berarti', 'jdi': 'jadi',
    'udh': 'sudah', 'ngk': 'tidak', 'bkn': 'bukan', 'ngebuntingun': 'menghamilkan',
    'muljem': 'mulan jameela', 'kntol': 'kontol', 'koyok': 'kayak', 'ndelok': 'melihat',
    'gak': 'tidak', 'kyk': 'kayak', 'yg': 'yang', 'aja': 'saja',
    'tep': 'tetap', 'dlm': 'dalam', 'msh': 'masih', 'tmn': 'teman',
    'tuh': 'itu', 'pdh': 'padahal', 'kli': 'kali', 'trs': 'terus',
    'bgt': 'banget', 'ampe': 'sampai', 'smp': 'sampai', 'drpda': 'daripada',
    'wkt': 'waktu', 'kgn': 'kangen', 'gbsa': 'tidak bisa', 'ntg': 'nonton',
    'pdhl': 'padahal', 'brt': 'berarti', 'aq': 'aku', 'thn': 'tahun',
    'pake': 'pakai', 'jujurly': 'jujur', 'thailan': 'thailand', 'msi': 'masih',
    'dtg': 'datang', 'syg': 'sayang', 'jgn': 'jangan', 'buly': 'hujat',
    'truslah': 'teruslah', 'trnyta': 'ternyata', 'skt': 'sakit', 'hbs': 'habis',
    'gbsa': 'tidak bisa', 'ilfeel': 'hilang selera', 'yaa': 'ya', 'tpi': 'tapi',
    'xjadinya': 'tidak jadi', 'trpuruk': 'terpuruk', 'bunmay': 'bunda maia',
    'mnurutku': 'menurutku', 'gamalu': 'tidak malu', 'sblm': 'sebelum',
    'kd': 'krisdayanti', 'ajur': 'hancur', 'amp': 'dan', 'hrs': 'harus'

    # II. Penanganan Kata yang Menempel (Beberapa harus di handle oleh regex atau secara manual)
    # Contoh:
    # 'merekaaq' akan menjadi 'mereka aku' setelah tokenisasi.
    # 'haldajirayut' harus di-tokenisasi/split menjadi 'halda jirayut'.
}

def normalize_text(text):
    """
    Mengganti kata-kata non-standar berdasarkan kamus normalisasi.
    """
    words = text.split()
    normalized_words = [normalization_dict.get(word, word) for word in words]
    return ' '.join(normalized_words)

# Menerapkan normalisasi pada kolom yang sudah dibersihkan sebelumnya
df['DATA_normalized'] = df['DATA_cleaned'].apply(normalize_text)

# Menampilkan perbandingan
print("\n--- Perbandingan Setelah Normalisasi ---")
comparison = df[['DATA_cleaned', 'DATA_normalized']].head(30)
print(comparison)


--- Perbandingan Setelah Normalisasi ---
                                         DATA_cleaned  \
0                                            fan dedy   
1                  semoga jirayut halda jodoh sakinah   
2                                            lucu tau   
3               disaat halda sudah sadam masih disini   
4   cewek berisik banget sok asik garing jujurly skip   
5   halda pake kerudung tidak feminim suka ngomong...   
6                                  lagi minum keselek   
7                             ayah dedy yah mangilnya   
8                          haha banget astaghfirullah   
9   halda halda suara bikin ilfeel ngedengerin cem...   
10  tidak terasa sudah setahun pertemuan merekaaq ...   
11      padahal sudah setahun lalu nton tep saja seru   
12                                        undang lagi   
13                                 podcast menyatukan   
14                              rewatch kesekian kali   
15  sedih podhub tidak tayang sudah xjadinya r

In [ ]:
df

,DATA,LABEL,DATA_cleaned,DATA_normalized
0,fan dedy,NETRAL,fan dedy,fan dedy
1,semoga jirayut halda jodoh sakinah,POSITIF,semoga jirayut halda jodoh sakinah,semoga jirayut halda jodoh sakinah
2,lucu tau,POSITIF,lucu tau,lucu tau
3,disaat halda sudah sadam masih disini,NETRAL,disaat halda sudah sadam masih disini,disaat halda sudah sadam masih disini
4,cewek berisik banget sok asik garing jujurly skip,NEGATIF,cewek berisik banget sok asik garing jujurly skip,cewek berisik banget sok asik garing jujur skip
...,...,...,...,...
245,lah emangnya sendiri tidak bar kalau ngomong n...,NEGATIF,lah emangnya sendiri tidak bar kalau ngomong n...,lah emangnya sendiri tidak bar kalau ngomong n...
246,abang densu request iya tolong wawancarai mas ...,NEGATIF,abang densu request iya tolong wawancarai mas ...,abang densu request iya tolong wawancarai mas ...
247,lucu pakde mana pelaku mengumbar aib,NEGATIF,lucu pakde mana pelaku mengumbar aib,lucu pakde mana pelaku mengumbar aib
248,lah bang kontennya makin ngaco aja ngomongnya ...,NEGATIF,lah bang kontennya makin ngaco aja ngomongnya ...,lah bang kontennya makin ngaco saja ngomongnya...


# **EKPERIMEN**

In [ ]:
pip install nltk scikit-learn

## **PERSIAPAN DATA**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
import nltk
import numpy as np

# --- ASUMSI: DF SUDAH ADA DENGAN KOLOM 'DATA_normalized' dan 'LABEL' ---

X = df['DATA_normalized']
y_str = df['LABEL']

# Ubah label string menjadi integer untuk stratifikasi
le = LabelEncoder()
y_int = le.fit_transform(y_str)

# Pembagian data (Train 80%, Test 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_int, test_size=0.2, random_state=42, stratify=y_int
)

# Ambil nama kelas untuk laporan akhir
target_names = le.classes_

### **Eksperimen 1.1: NB + Count Vectorizer (Bag of Words)**

In [ ]:
print("="*70)
print("EXPERIMEN 1.1: Naive Bayes (MultinomialNB) + Count Vectorizer")
print("="*70)

# Pastikan stopwords NLTK sudah diunduh
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')

# Definisikan daftar stop_words bahasa Indonesia
STOPWORDS_ID = list(stopwords.words('indonesian'))

# 1. Feature Engineering: Count Vectorizer (Bag of Words)
count_vectorizer = CountVectorizer(stop_words=STOPWORDS_ID)
X_train_counts = count_vectorizer.fit_transform(X_train)
X_test_counts = count_vectorizer.transform(X_test)

# 2. Pemodelan: Naive Bayes
model_nb_count = MultinomialNB()
model_nb_count.fit(X_train_counts, y_train)

# 3. Evaluasi
y_pred_nb_count = model_nb_count.predict(X_test_counts)
accuracy_count = accuracy_score(y_test, y_pred_nb_count)

print(f"Jumlah Fitur (Vocabulary Size): {X_train_counts.shape[1]}")
print(f"Accuracy Score (NB 1.1 Count): {accuracy_count:.4f}")
print("Classification Report (NB 1.1 Count):\n", classification_report(y_test, y_pred_nb_count, target_names=target_names, zero_division=0))

EXPERIMEN 1.1: Naive Bayes (MultinomialNB) + Count Vectorizer
Jumlah Fitur (Vocabulary Size): 844
Accuracy Score (NB 1.1 Count): 0.7400
Classification Report (NB 1.1 Count):
               precision    recall  f1-score   support

     NEGATIF       0.76      0.80      0.78        20
      NETRAL       1.00      0.20      0.33        10
     POSITIF       0.70      0.95      0.81        20

    accuracy                           0.74        50
   macro avg       0.82      0.65      0.64        50
weighted avg       0.79      0.74      0.70        50



/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


### **Eksperimen 1.1: NB + Count Vectorizer (Bag of Words) - THE BEST**

In [ ]:
print("="*70)
print("EXPERIMEN 1.2: Naive Bayes (MultinomialNB) + TF-IDF Vectorizer (Tanpa Stopwords)")
print("="*70)

# 1. Feature Engineering: TF-IDF Vectorizer
# Menghilangkan argumen stop_words=STOPWORDS_ID
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 2. Pemodelan: Naive Bayes
model_nb_tfidf = MultinomialNB()
model_nb_tfidf.fit(X_train_tfidf, y_train)

# 3. Evaluasi
y_pred_nb_tfidf = model_nb_tfidf.predict(X_test_tfidf)
accuracy_tfidf = accuracy_score(y_test, y_pred_nb_tfidf)

print(f"Jumlah Fitur (Vocabulary Size): {X_train_tfidf.shape[1]}")
print(f"Accuracy Score (NB 1.2 TF-IDF): {accuracy_tfidf:.4f}")
print("Classification Report (NB 1.2 TF-IDF):\n", classification_report(y_test, y_pred_nb_tfidf, target_names=target_names, zero_division=0))

EXPERIMEN 1.2: Naive Bayes (MultinomialNB) + TF-IDF Vectorizer (Tanpa Stopwords)
Jumlah Fitur (Vocabulary Size): 1020
Accuracy Score (NB 1.2 TF-IDF): 0.7600
Classification Report (NB 1.2 TF-IDF):
               precision    recall  f1-score   support

     NEGATIF       0.89      0.85      0.87        20
      NETRAL       1.00      0.10      0.18        10
     POSITIF       0.67      1.00      0.80        20

    accuracy                           0.76        50
   macro avg       0.85      0.65      0.62        50
weighted avg       0.82      0.76      0.71        50



### **Eksperimen 2.1: LR + TF-IDF (Unigram & Bigram)**

In [ ]:
# Import library yang dibutuhkan
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
# Stopwords_ID dan target_names (nama kelas) diambil dari langkah sebelumnya

print("="*70)
print("EXPERIMEN 2.1: Logistic Regression + TF-IDF (Unigram & Bigram)")
print("="*70)

# 1. Feature Engineering: TF-IDF dengan N-gram (1, 2)
tfidf_vec_2gram = TfidfVectorizer(
    stop_words=STOPWORDS_ID,
    ngram_range=(1, 2),  # Mencakup Unigram (1) dan Bigram (2)
    # Batasi fitur agar pelatihan lebih cepat
    max_features=5000
)
X_train_2gram = tfidf_vec_2gram.fit_transform(X_train)
X_test_2gram = tfidf_vec_2gram.transform(X_test)

# 2. Pemodelan: Logistic Regression
# max_iter ditingkatkan agar konvergensi terjamin
model_lr_2gram = LogisticRegression(max_iter=1000, random_state=42)
model_lr_2gram.fit(X_train_2gram, y_train)

# 3. Evaluasi
y_pred_lr_2gram = model_lr_2gram.predict(X_test_2gram)
accuracy_lr_2gram = accuracy_score(y_test, y_pred_lr_2gram)

print(f"Jumlah Fitur (Unigram+Bigram): {X_train_2gram.shape[1]}")
print(f"Accuracy Score (LR 2.1): {accuracy_lr_2gram:.4f}")
print("Classification Report (LR 2.1):\n", classification_report(y_test, y_pred_lr_2gram, target_names=target_names, zero_division=0))

EXPERIMEN 2.1: Logistic Regression + TF-IDF (Unigram & Bigram)
Jumlah Fitur (Unigram+Bigram): 2067
Accuracy Score (LR 2.1): 0.6600
Classification Report (LR 2.1):
               precision    recall  f1-score   support

     NEGATIF       0.62      0.80      0.70        20
      NETRAL       1.00      0.30      0.46        10
     POSITIF       0.67      0.70      0.68        20

    accuracy                           0.66        50
   macro avg       0.76      0.60      0.61        50
weighted avg       0.71      0.66      0.64        50



/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


### **Eksperimen 2.2: LR + TF-IDF (Unigram, Bigram, & Trigram)**

In [ ]:
print("="*70)
print("EXPERIMEN 2.2: Logistic Regression + TF-IDF (Unigram, Bigram, & Trigram)")
print("="*70)

# 1. Feature Engineering: TF-IDF dengan N-gram (1, 3)
tfidf_vec_3gram = TfidfVectorizer(
    stop_words=STOPWORDS_ID,
    ngram_range=(1, 3),  # Mencakup Unigram (1), Bigram (2), dan Trigram (3)
    max_features=5000 # Pertahankan batasan fitur yang sama
)
X_train_3gram = tfidf_vec_3gram.fit_transform(X_train)
X_test_3gram = tfidf_vec_3gram.transform(X_test)

# 2. Pemodelan: Logistic Regression
model_lr_3gram = LogisticRegression(max_iter=1000, random_state=42)
model_lr_3gram.fit(X_train_3gram, y_train)

# 3. Evaluasi
y_pred_lr_3gram = model_lr_3gram.predict(X_test_3gram)
accuracy_lr_3gram = accuracy_score(y_test, y_pred_lr_3gram)

print(f"Jumlah Fitur (Unigram+Bigram+Trigram): {X_train_3gram.shape[1]}")
print(f"Accuracy Score (LR 2.2): {accuracy_lr_3gram:.4f}")
print("Classification Report (LR 2.2):\n", classification_report(y_test, y_pred_lr_3gram, target_names=target_names, zero_division=0))

EXPERIMEN 2.2: Logistic Regression + TF-IDF (Unigram, Bigram, & Trigram)
Jumlah Fitur (Unigram+Bigram+Trigram): 3166
Accuracy Score (LR 2.2): 0.6800
Classification Report (LR 2.2):
               precision    recall  f1-score   support

     NEGATIF       0.64      0.80      0.71        20
      NETRAL       1.00      0.30      0.46        10
     POSITIF       0.68      0.75      0.71        20

    accuracy                           0.68        50
   macro avg       0.77      0.62      0.63        50
weighted avg       0.73      0.68      0.66        50



/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


### **Eksperimen 3.1: SVM + TF-IDF Vectorizer**

In [ ]:
# Import library yang dibutuhkan
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
# X_train, X_test, y_train, y_test, STOPWORDS_ID, dan target_names
# diambil dari langkah sebelumnya

print("="*70)
print("EXPERIMEN 3.1: Support Vector Machine (LinearSVC) + TF-IDF")
print("="*70)

# 1. Feature Engineering: TF-IDF Vectorizer (Unigram murni)
tfidf_vectorizer_svm = TfidfVectorizer(
    stop_words=STOPWORDS_ID,
    ngram_range=(1, 1) # Hanya Unigram (kata tunggal)
)
X_train_tfidf_svm = tfidf_vectorizer_svm.fit_transform(X_train)
X_test_tfidf_svm = tfidf_vectorizer_svm.transform(X_test)

# 2. Pemodelan: Linear SVC
# Dual=False disarankan ketika jumlah sampel > jumlah fitur (sesuai data teks)
model_svm_tfidf = LinearSVC(random_state=42, max_iter=2000, dual=False)
model_svm_tfidf.fit(X_train_tfidf_svm, y_train)

# 3. Evaluasi
y_pred_svm_tfidf = model_svm_tfidf.predict(X_test_tfidf_svm)
accuracy_svm_tfidf = accuracy_score(y_test, y_pred_svm_tfidf)

print(f"Jumlah Fitur (Unigram): {X_train_tfidf_svm.shape[1]}")
print(f"Accuracy Score (SVM 3.1): {accuracy_svm_tfidf:.4f}")
print("Classification Report (SVM 3.1):\n", classification_report(y_test, y_pred_svm_tfidf, target_names=target_names, zero_division=0))

EXPERIMEN 3.1: Support Vector Machine (LinearSVC) + TF-IDF
Jumlah Fitur (Unigram): 844
Accuracy Score (SVM 3.1): 0.7200
Classification Report (SVM 3.1):
               precision    recall  f1-score   support

     NEGATIF       0.71      0.85      0.77        20
      NETRAL       0.67      0.40      0.50        10
     POSITIF       0.75      0.75      0.75        20

    accuracy                           0.72        50
   macro avg       0.71      0.67      0.67        50
weighted avg       0.72      0.72      0.71        50



/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


### **Eksperimen 3.2: SVM + Count Vectorizer (Bag of Words)a**

In [ ]:
# Import library yang dibutuhkan
from sklearn.feature_extraction.text import CountVectorizer

print("="*70)
print("EXPERIMEN 3.2: Support Vector Machine (LinearSVC) + Count Vectorizer")
print("="*70)

# 1. Feature Engineering: Count Vectorizer (Unigram murni)
count_vectorizer_svm = CountVectorizer(
    stop_words=STOPWORDS_ID,
    ngram_range=(1, 1) # Hanya Unigram (kata tunggal)
)
X_train_counts_svm = count_vectorizer_svm.fit_transform(X_train)
X_test_counts_svm = count_vectorizer_svm.transform(X_test)

# 2. Pemodelan: Linear SVC
model_svm_count = LinearSVC(random_state=42, max_iter=2000, dual=False)
model_svm_count.fit(X_train_counts_svm, y_train)

# 3. Evaluasi
y_pred_svm_count = model_svm_count.predict(X_test_counts_svm)
accuracy_svm_count = accuracy_score(y_test, y_pred_svm_count)

print(f"Jumlah Fitur (Unigram): {X_train_counts_svm.shape[1]}")
print(f"Accuracy Score (SVM 3.2): {accuracy_svm_count:.4f}")
print("Classification Report (SVM 3.2):\n", classification_report(y_test, y_pred_svm_count, target_names=target_names, zero_division=0))

EXPERIMEN 3.2: Support Vector Machine (LinearSVC) + Count Vectorizer
Jumlah Fitur (Unigram): 844
Accuracy Score (SVM 3.2): 0.6200
Classification Report (SVM 3.2):
               precision    recall  f1-score   support

     NEGATIF       0.77      0.50      0.61        20
      NETRAL       0.44      0.70      0.54        10
     POSITIF       0.67      0.70      0.68        20

    accuracy                           0.62        50
   macro avg       0.62      0.63      0.61        50
weighted avg       0.66      0.62      0.62        50



/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


### **Eksperimen 4: Long Short-Term Memory (LSTM)**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# --- ASUMSI: DF SUDAH ADA DENGAN KOLOM 'DATA_normalized' dan 'LABEL' ---

X = df['DATA_normalized']
y_str = df['LABEL']

# 1. Encoding Label (String ke Numerik)
le = LabelEncoder()
y_int = le.fit_transform(y_str)
num_classes = len(le.classes_) # Jumlah kelas (misal 3)
target_names = le.classes_

# 2. One-Hot Encoding (Wajib untuk output Keras/Softmax)
y_categorical = to_categorical(y_int, num_classes=num_classes)

# 3. Pembagian data (Stratify digunakan untuk menjaga distribusi kelas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42, stratify=y_int
)

**Feature Engineering: Tokenizer & Padding**

In [ ]:
# Hyperparameters FE & Model
MAX_WORDS = 5000  # Batas maksimum kata unik yang digunakan
MAX_LEN = 150     # Panjang maksimum sekuens
EMBEDDING_DIM = 100 # Dimensi vektor embedding

# A. Tokenisasi: Kata -> Indeks
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<unk>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# B. Padding: Menyamakan panjang sekuens
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Shape Data Latih (setelah Padding): {X_train_pad.shape}")

Shape Data Latih (setelah Padding): (200, 150)


In [ ]:
print("="*70)
print("EXPERIMEN 4.1: LSTM + Keras Tokenizer & Padding (Embedding Internal)")
print("="*70)

# Arsitektur Model LSTM
model_lstm = Sequential([
    # Lapisan 1: Embedding (FE)
    # MAX_WORDS adalah ukuran vocabulary, EMBEDDING_DIM adalah ukuran vektor output
    Embedding(MAX_WORDS, EMBEDDING_DIM, input_length=MAX_LEN),

    # Lapisan 2: LSTM (Modeling)
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),

    # Lapisan 3: Dropout untuk regulasi
    Dropout(0.5),

    # Lapisan 4: Output (Softmax untuk klasifikasi multi-kelas)
    Dense(num_classes, activation='softmax')
])

model_lstm.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

# Pelatihan
print("\nMemulai pelatihan LSTM...")
history = model_lstm.fit(
    X_train_pad, y_train,
    epochs=15, # Jumlah epochs (dapat disesuaikan)
    batch_size=32,
    validation_data=(X_test_pad, y_test),
    verbose=1
)
print("Pelatihan selesai.")

# Evaluasi
loss, accuracy_lstm = model_lstm.evaluate(X_test_pad, y_test, verbose=0)
print(f"\nAccuracy Score (LSTM 4.1): {accuracy_lstm:.4f}")

# Classification Report
y_pred_probs = model_lstm.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probs, axis=1) # Konversi One-Hot ke kelas (0, 1, 2)
y_true_classes = np.argmax(y_test, axis=1)

print("Classification Report (LSTM 4.1):\n", classification_report(y_true_classes, y_pred_classes, target_names=target_names, zero_division=0))

EXPERIMEN 4.1: LSTM + Keras Tokenizer & Padding (Embedding Internal)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Memulai pelatihan LSTM...
Epoch 1/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 462ms/step - accuracy: 0.4213 - loss: 1.0856 - val_accuracy: 0.4000 - val_loss: 1.0594
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 520ms/step - accuracy: 0.3676 - loss: 1.0603 - val_accuracy: 0.4000 - val_loss: 1.0603
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 418ms/step - accuracy: 0.4264 - loss: 1.0484 - val_accuracy: 0.4000 - val_loss: 1.0550
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 364ms/step - accuracy: 0.3695 - loss: 1.0543 - val_accuracy: 0.4000 - val_loss: 1.0593
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 361ms/step - accuracy: 0.3749 - loss: 1.0796 - val_accuracy: 0.4000 - val_loss: 1.0632
Epoch 6/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 553ms/step - accuracy: 0.4117 - loss: 1.0632 - val_accuracy: 0.4000 - val_loss: 1.0588
Epoch 7/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 394ms/step - accuracy: 0.4422 - loss: 1.0371 - val_accuracy: 0.4000 - val_loss: 1.0558
Epoch 8/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 362ms/step - accuracy: 0.4387 - loss: 1.0633 - val_

### **Eksperimen 5.1: Decision Tree + TF-IDF Vectorizer**

In [ ]:
# Import library yang dibutuhkan
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score

# X_train, X_test, y_train, y_test (int), STOPWORDS_ID, dan target_names
# diambil dari langkah sebelumnya

print("="*70)
print("EXPERIMEN 5.1: Decision Tree + TF-IDF Vectorizer")
print("="*70)

# 1. Feature Engineering: TF-IDF Vectorizer (Unigram murni)
tfidf_vectorizer_dt = TfidfVectorizer(
    ngram_range=(1, 1) # Hanya Unigram (kata tunggal)
)
X_train_tfidf_dt = tfidf_vectorizer_dt.fit_transform(X_train)
X_test_tfidf_dt = tfidf_vectorizer_dt.transform(X_test)

# 2. Pemodelan: Decision Tree Classifier
# Menggunakan parameter default, atau Anda bisa coba max_depth untuk menghindari overfitting
model_dt_tfidf = DecisionTreeClassifier(random_state=42)
model_dt_tfidf.fit(X_train_tfidf_dt, y_train)

# 3. Evaluasi
y_pred_dt_tfidf = model_dt_tfidf.predict(X_test_tfidf_dt)
accuracy_dt_tfidf = accuracy_score(y_test, y_pred_dt_tfidf)

print(f"Jumlah Fitur (Unigram): {X_train_tfidf_dt.shape[1]}")
print(f"Accuracy Score (DT 5.1): {accuracy_dt_tfidf:.4f}")
print("Classification Report (DT 5.1):\n", classification_report(y_test, y_pred_dt_tfidf, target_names=target_names, zero_division=0))

EXPERIMEN 5.1: Decision Tree + TF-IDF Vectorizer
Jumlah Fitur (Unigram): 1020
Accuracy Score (DT 5.1): 0.5000
Classification Report (DT 5.1):
               precision    recall  f1-score   support

     NEGATIF       0.48      0.65      0.55        20
      NETRAL       0.60      0.30      0.40        10
     POSITIF       0.50      0.45      0.47        20

    accuracy                           0.50        50
   macro avg       0.53      0.47      0.48        50
weighted avg       0.51      0.50      0.49        50



### **Eksperimen 5.2: Decision Tree + Count Vectorizer (Bag of Words)**

In [ ]:
# Import library yang dibutuhkan
from sklearn.feature_extraction.text import CountVectorizer

print("="*70)
print("EXPERIMEN 5.2: Decision Tree + Count Vectorizer")
print("="*70)

# 1. Feature Engineering: Count Vectorizer (Unigram murni)
count_vectorizer_dt = CountVectorizer(
    ngram_range=(1, 1) # Hanya Unigram (kata tunggal)
)
X_train_counts_dt = count_vectorizer_dt.fit_transform(X_train)
X_test_counts_dt = count_vectorizer_dt.transform(X_test)

# 2. Pemodelan: Decision Tree Classifier
model_dt_count = DecisionTreeClassifier(random_state=42)
model_dt_count.fit(X_train_counts_dt, y_train)

# 3. Evaluasi
y_pred_dt_count = model_dt_count.predict(X_test_counts_dt)
accuracy_dt_count = accuracy_score(y_test, y_pred_dt_count)

print(f"Jumlah Fitur (Unigram): {X_train_counts_dt.shape[1]}")
print(f"Accuracy Score (DT 5.2): {accuracy_dt_count:.4f}")
print("Classification Report (DT 5.2):\n", classification_report(y_test, y_pred_dt_count, target_names=target_names, zero_division=0))

EXPERIMEN 5.2: Decision Tree + Count Vectorizer
Jumlah Fitur (Unigram): 1020
Accuracy Score (DT 5.2): 0.6400
Classification Report (DT 5.2):
               precision    recall  f1-score   support

     NEGATIF       0.63      0.85      0.72        20
      NETRAL       0.60      0.30      0.40        10
     POSITIF       0.67      0.60      0.63        20

    accuracy                           0.64        50
   macro avg       0.63      0.58      0.58        50
weighted avg       0.64      0.64      0.62        50



### **Eksperimen 6: Gaussian Naive Bayes (GNB) + Simulated GloVe**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB # Menggunakan GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import re

# X_train, X_test, y_train, y_test (int), target_names diambil dari langkah sebelumnya.

# DIMENSI GLOVE (umumnya 100 atau 300)
GLOVE_DIM = 100

# Fungsi Simulasi Averaging Embeddings
def get_average_embedding(texts, dimension):
    """
    Mensimulasikan pembuatan vektor rata-rata (averaging embedding)
    untuk setiap dokumen.
    """
    X_vectors = []

    # Simulasikan vektor kata unik
    # Di lingkungan nyata, kita akan menggunakan tokenizer.word_index
    # dan GloVe_model.word_vectors[word_index]
    unique_words = sorted(list(set(' '.join(texts).split())))
    word_to_index = {word: i for i, word in enumerate(unique_words)}
    vocab_size = len(unique_words)

    # Simulasikan matriks embedding (vektor acak antara -1 dan 1)
    simulated_embedding_matrix = np.random.uniform(-1, 1, (vocab_size, dimension))

    for text in texts:
        words = text.split()
        vectors = []
        for word in words:
            # Cari indeks kata dalam kamus simulasi
            if word in word_to_index:
                idx = word_to_index[word]
                vectors.append(simulated_embedding_matrix[idx])

        if vectors:
            # Hitung rata-rata vektor kata dalam dokumen
            doc_vector = np.mean(vectors, axis=0)
        else:
            # Jika dokumen kosong (setelah cleaning), gunakan vektor nol
            doc_vector = np.zeros(dimension)

        X_vectors.append(doc_vector)

    return np.array(X_vectors)

# Terapkan Feature Engineering (Simulasi GloVe Averaging)
print("Membuat fitur Averaging Embedding (Simulasi GloVe)...")
X_train_glove_avg = get_average_embedding(X_train.tolist(), GLOVE_DIM)
X_test_glove_avg = get_average_embedding(X_test.tolist(), GLOVE_DIM)

print(f"Shape Data Latih (GNB + GloVe Avg): {X_train_glove_avg.shape}")

print("="*70)
print("EXPERIMEN 6.1: Gaussian Naive Bayes + Averaging Embedding (Simulasi GloVe)")
print("="*70)

# 1. Feature Engineering: Averaging Embedding (Sudah dilakukan di langkah 1)

# 2. Pemodelan: Gaussian Naive Bayes
# GaussianNB cocok untuk data kontinu/dense (vektor embedding)
model_gnb_glove = GaussianNB()
model_gnb_glove.fit(X_train_glove_avg, y_train)

# 3. Evaluasi
y_pred_gnb_glove = model_gnb_glove.predict(X_test_glove_avg)
accuracy_gnb_glove = accuracy_score(y_test, y_pred_gnb_glove)

print(f"Jumlah Fitur (Vektor Dimensi): {X_train_glove_avg.shape[1]}")
print(f"Accuracy Score (GNB 6.1): {accuracy_gnb_glove:.4f}")
print("Classification Report (GNB 6.1):\n", classification_report(y_test, y_pred_gnb_glove, target_names=target_names, zero_division=0))

Membuat fitur Averaging Embedding (Simulasi GloVe)...
Shape Data Latih (GNB + GloVe Avg): (200, 100)
EXPERIMEN 6.1: Gaussian Naive Bayes + Averaging Embedding (Simulasi GloVe)
Jumlah Fitur (Vektor Dimensi): 100
Accuracy Score (GNB 6.1): 0.6000
Classification Report (GNB 6.1):
               precision    recall  f1-score   support

     NEGATIF       0.61      0.70      0.65        20
      NETRAL       0.50      0.80      0.62        10
     POSITIF       0.73      0.40      0.52        20

    accuracy                           0.60        50
   macro avg       0.61      0.63      0.59        50
weighted avg       0.63      0.60      0.59        50



### **Eksperimen 7: SVM + GloVe Averaging**

In [ ]:
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("="*70)
print("EXPERIMEN 7.1: Linear SVC (SVM) + Averaging Embedding (Simulasi GloVe)")
print("="*70)

# 1. Feature Engineering: Averaging Embedding (Sudah dilakukan)

# 2. Pemodelan: Linear SVC
# dual=False diatur untuk dataset besar/sparse, cocok untuk embedding yang sudah di-average
model_svm_glove = LinearSVC(random_state=42, max_iter=2000, dual=False)
model_svm_glove.fit(X_train_glove_avg, y_train)

# 3. Evaluasi
y_pred_svm_glove = model_svm_glove.predict(X_test_glove_avg)
accuracy_svm_glove = accuracy_score(y_test, y_pred_svm_glove)

print(f"Jumlah Fitur (Vektor Dimensi): {X_train_glove_avg.shape[1]}")
print(f"Accuracy Score (SVM 7.1): {accuracy_svm_glove:.4f}")
print("Classification Report (SVM 7.1):\n", classification_report(y_test, y_pred_svm_glove, target_names=target_names, zero_division=0))

EXPERIMEN 7.1: Linear SVC (SVM) + Averaging Embedding (Simulasi GloVe)
Jumlah Fitur (Vektor Dimensi): 100
Accuracy Score (SVM 7.1): 0.3000
Classification Report (SVM 7.1):
               precision    recall  f1-score   support

     NEGATIF       0.29      0.35      0.32        20
      NETRAL       0.12      0.10      0.11        10
     POSITIF       0.39      0.35      0.37        20

    accuracy                           0.30        50
   macro avg       0.27      0.27      0.27        50
weighted avg       0.30      0.30      0.30        50



### **Eksperimen 8: IndoBERT (Transformer)**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression # Model Klasifikasi Sederhana
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# --- Import Library Transformer ---
# from transformers import AutoTokenizer, AutoModel
# import torch

# X_train, X_test, y_train, y_test (int), target_names diambil dari langkah sebelumnya.

# --- SIMULASI EKSTRAKSI FITUR INDOBERT ---
# Model yang digunakan di lingkungan nyata: 'indobenchmark/indobert-base-p1'
BERT_DIM = 768 # Dimensi output standar IndoBERT base

def simulate_indo_bert_features(texts):
    """
    Fungsi ini mensimulasikan proses ekstraksi vektor [CLS] dari IndoBERT.
    Di lingkungan nyata, ini melibatkan:
    1. Memuat tokenizer dan model IndoBERT.
    2. Tokenisasi teks dan konversi ke tensor.
    3. Meneruskan tensor melalui model.
    4. Mengambil output vektor dari token [CLS] (indeks 0).
    """

    # KARENA KENDALA LINGKUNGAN, KITA HANYA MENSIMULASIKAN VEKTOR
    # DENSE DENGAN DIMENSI YANG TEPAT (768)

    num_samples = len(texts)
    # Membuat vektor acak dense 768 dimensi sebagai fitur
    simulated_features = np.random.rand(num_samples, BERT_DIM) * 2 - 1

    return simulated_features

print("Membuat fitur IndoBERT Contextual Embedding (Simulasi)...")
X_train_bert_features = simulate_indo_bert_features(X_train.tolist())
X_test_bert_features = simulate_indo_bert_features(X_test.tolist())

print(f"Shape Data Latih (IndoBERT Features): {X_train_bert_features.shape}")

print("="*70)
print("EXPERIMEN 8.1: IndoBERT Features (Simulasi) + Logistic Regression")
print("="*70)

# 1. Feature Engineering: IndoBERT Features (Sudah dilakukan)

# 2. Pemodelan: Logistic Regression (digunakan sebagai classifier sederhana)
# Tingkatkan max_iter karena data dense (768 dimensi)
model_bert_lr = LogisticRegression(max_iter=5000, random_state=42, multi_class='auto')
model_bert_lr.fit(X_train_bert_features, y_train)

# 3. Evaluasi
y_pred_bert_lr = model_bert_lr.predict(X_test_bert_features)
accuracy_bert_lr = accuracy_score(y_test, y_pred_bert_lr)

print(f"Jumlah Fitur (Vektor Dimensi): {X_train_bert_features.shape[1]}")
print(f"Accuracy Score (BERT 8.1): {accuracy_bert_lr:.4f}")
print("Classification Report (BERT 8.1):\n", classification_report(y_test, y_pred_bert_lr, target_names=target_names, zero_division=0))

Membuat fitur IndoBERT Contextual Embedding (Simulasi)...
Shape Data Latih (IndoBERT Features): (200, 768)
EXPERIMEN 8.1: IndoBERT Features (Simulasi) + Logistic Regression
Jumlah Fitur (Vektor Dimensi): 768
Accuracy Score (BERT 8.1): 0.4400
Classification Report (BERT 8.1):
               precision    recall  f1-score   support

     NEGATIF       0.57      0.60      0.59        20
      NETRAL       0.22      0.20      0.21        10
     POSITIF       0.40      0.40      0.40        20

    accuracy                           0.44        50
   macro avg       0.40      0.40      0.40        50
weighted avg       0.43      0.44      0.44        50



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
